# Logistic Regression: сохранение и загрузка модели

In [37]:
from pathlib import Path

import joblib
import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report, f1_score, precision_score, recall_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

In [38]:
DATA_PATH = Path("../data/raw/UCI_Credit_Card.csv")
MODEL_PATH = Path("../models/logistic_regression_model.joblib")

In [39]:
df = pd.read_csv(DATA_PATH)

print(df.shape)
df.head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default.payment.next.month
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [40]:
target = "default.payment.next.month"

X = df.drop(columns=["ID", target])
y = df[target]

print("Количество признаков:", X.shape[1])
print("Доля дефолтов:", y.mean().round(3))

Количество признаков: 23
Доля дефолтов: 0.221


In [41]:
# Делаем три выборки: train для обучения, validation для подбора порога, test для финальной проверки
X_train, X_temp, y_train, y_temp = train_test_split(
    X,
    y,
    test_size=0.4,
    random_state=42,
    stratify=y
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print("Train:", X_train.shape)
print("Validation:", X_val.shape)
print("Test:", X_test.shape)

Train: (18000, 23)
Validation: (6000, 23)
Test: (6000, 23)


In [42]:
# У дефолтов меньше наблюдений, поэтому немного повышаем вес класса 1
model = Pipeline(steps=[
    ("scaler", StandardScaler()),
    ("classifier", LogisticRegression(max_iter=1000, class_weight={0: 1, 1: 2}, random_state=42))
])

model.fit(X_train, y_train)

Pipeline(steps=[('scaler', StandardScaler()),
                ('classifier',
                 LogisticRegression(class_weight={0: 1, 1: 2}, max_iter=1000,
                                    random_state=42))])

In [43]:
# Стандартный порог 0.5 давал низкий recall, поэтому подбираем порог по F1 на validation
val_proba = model.predict_proba(X_val)[:, 1]

rows = []
for threshold in [i / 100 for i in range(10, 71)]:
    val_pred = (val_proba >= threshold).astype(int)
    rows.append({
        "threshold": threshold,
        "precision": precision_score(y_val, val_pred, zero_division=0),
        "recall": recall_score(y_val, val_pred),
        "f1": f1_score(y_val, val_pred)
    })

threshold_results = pd.DataFrame(rows)
threshold_results.sort_values("f1", ascending=False).head(10)

,threshold,precision,recall,f1
32,0.42,0.502720,0.487566,0.495027
30,0.40,0.464901,0.529013,0.494889
33,0.43,0.519199,0.468726,0.492673
31,0.41,0.482310,0.503391,0.492625
34,0.44,0.534925,0.455916,0.492270
29,0.39,0.437162,0.547852,0.486288
35,0.45,0.541667,0.440844,0.486082
36,0.46,0.552224,0.430294,0.483693
38,0.48,0.572614,0.415976,0.481886
28,0.38,0.413925,0.568953,0.479213


In [44]:
best_threshold = threshold_results.sort_values("f1", ascending=False).iloc[0]["threshold"]
best_threshold

np.float64(0.42)

In [45]:
# На test проверяем качество только один раз, уже с выбранным порогом
test_proba = model.predict_proba(X_test)[:, 1]
y_pred = (test_proba >= best_threshold).astype(int)

print("Threshold:", round(best_threshold, 2))
print("Accuracy:", round(accuracy_score(y_test, y_pred), 3))
print("Precision:", round(precision_score(y_test, y_pred), 3))
print("Recall:", round(recall_score(y_test, y_pred), 3))
print("F1-score:", round(f1_score(y_test, y_pred), 3))
print()
print(classification_report(y_test, y_pred))

Threshold: 0.42
Accuracy: 0.786
Precision: 0.517
Recall: 0.512
F1-score: 0.515

              precision    recall  f1-score   support

           0       0.86      0.86      0.86      4673
           1       0.52      0.51      0.51      1327

    accuracy                           0.79      6000
   macro avg       0.69      0.69      0.69      6000
weighted avg       0.79      0.79      0.79      6000



In [46]:
# Сохраняем модель вместе с порогом и списком признаков для будущего инференса
model_bundle = {
    "model": model,
    "threshold": float(best_threshold),
    "features": list(X.columns)
}

MODEL_PATH.parent.mkdir(parents=True, exist_ok=True)
joblib.dump(model_bundle, MODEL_PATH)

print("Модель сохранена:", MODEL_PATH)

Модель сохранена: ../models/logistic_regression_model.joblib


In [47]:
def load_model(path=MODEL_PATH):
    return joblib.load(path)

loaded_model = load_model()
loaded_model["threshold"]

0.42

In [48]:
client = X_test.iloc[[0]]

probability = loaded_model["model"].predict_proba(client)[0][1]
prediction = int(probability >= loaded_model["threshold"])

print("Прогноз класса:", prediction)
print("Вероятность дефолта:", round(probability, 3))

Прогноз класса: 1
Вероятность дефолта: 0.745


In [49]:
def predict_default(client_data, path=MODEL_PATH):
    model_bundle = load_model(path)
    client_df = pd.DataFrame(client_data)

    # Оставляем признаки в том же порядке, что был при обучении
    client_df = client_df[model_bundle["features"]]

    probability = model_bundle["model"].predict_proba(client_df)[0][1]
    prediction = int(probability >= model_bundle["threshold"])

    return {
        "prediction": prediction,
        "default_probability": float(probability)
    }